# Pair Benchmark: Constructing the Human Reference

This notebook prepares a reproducible visual benchmark for selecting a clustering method and its
parameters. The objective is to form high-purity groups of visually recurrent fleuron candidates
while letting unrelated crops remain unassigned.

Because the corpus carries no fleuron-family labels, visual judgement supplies the semantic
reference. It is combined with quantitative measures: pair precision, assigned coverage, noise rate,
cluster-size distribution, stability, and recurrence across scans and inferred books. UMAP is not
used as a clustering input.

---

**Reads** the feature matrix · **Writes** `pair_review/`, 180 labelled pairs · **Chapter README** §3

## 1. Configuration

In [1]:
import hashlib
import json
import math
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

RANDOM_SEED = 42
random_generator = np.random.default_rng(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd().resolve()
assert NOTEBOOK_DIR.name == '2_clustering', 'Run this notebook from its own folder, 2_clustering/.'
PROJECT_DIR = NOTEBOOK_DIR.parent
# ---------------------------------------------------------------------------
# Output versioning.
#   RUN_TAG = ''          reproduces the frozen thesis run (read-only guard on)
#   RUN_TAG = '_rerun1'   writes to a parallel directory, leaving the frozen
#                         artifacts untouched
# Input paths always point at the frozen artifacts, so a re-run is reproducible
# rather than self-referential.
# ---------------------------------------------------------------------------
RUN_TAG = '_rerun1'
ALLOW_OVERWRITE = False


def guarded_mkdir(path):
    """Create an output directory, refusing to overwrite a populated one."""
    if path.exists() and any(path.iterdir()) and not (RUN_TAG or ALLOW_OVERWRITE):
        raise RuntimeError(
            f'{path} already contains results. Set RUN_TAG to write elsewhere, '
            f'or ALLOW_OVERWRITE = True to replace them.')
    path.mkdir(parents=True, exist_ok=True)
    return path

FEATURE_RUN_ID = 'all_regions_v1_minside24_dinov2_vitb14_binarized_v1'
FEATURE_DIR = PROJECT_DIR / 'feature_extraction_outputs' / FEATURE_RUN_ID
FEATURES_PATH = FEATURE_DIR / 'dino_features_binarized.npy'
FEATURE_MANIFEST_PATH = FEATURE_DIR / 'features_manifest.csv'

OUTPUT_RUN_ID = f'{FEATURE_RUN_ID}_clustering_benchmark_v1'
OUTPUT_DIR = PROJECT_DIR / '2_clustering_outputs' / OUTPUT_RUN_ID
PAIR_REVIEW_DIR = OUTPUT_DIR / f'pair_review{RUN_TAG}'
SHEETS_DIR = PAIR_REVIEW_DIR / 'sheets'
PAIR_CANDIDATES_PATH = PAIR_REVIEW_DIR / 'pair_candidates.csv'
PAIR_LABELS_PATH = PAIR_REVIEW_DIR / 'pair_review_labels.csv'
PAIR_SUMMARY_PATH = PAIR_REVIEW_DIR / 'pair_sampling_summary.csv'
BOOK_SUMMARY_PATH = OUTPUT_DIR / 'inferred_book_summary.csv'
PROTOCOL_PATH = OUTPUT_DIR / 'benchmark_protocol.json'
INSTRUCTIONS_PATH = PAIR_REVIEW_DIR / 'README.md'

SIMILARITY_BINS = [
    ('0.80-0.85', 0.80, 0.85),
    ('0.85-0.90', 0.85, 0.90),
    ('0.90-0.93', 0.90, 0.93),
    ('0.93-0.95', 0.93, 0.95),
    ('0.95-0.97', 0.95, 0.97),
    ('0.97-1.00', 0.97, 1.00001),
]
PAIRS_PER_BIN = 30
CALIBRATION_PAIRS_PER_BIN = 20
RANDOM_PAIR_POOL = 500_000
QUERY_COUNT = 4_000
NEIGHBORS_PER_QUERY = 50
PAIRS_PER_SHEET = 12
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert FEATURES_PATH.exists() and FEATURE_MANIFEST_PATH.exists()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
guarded_mkdir(PAIR_REVIEW_DIR)
SHEETS_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()
    with file_path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

print('Feature run :', FEATURE_RUN_ID)
print('Output      :', OUTPUT_DIR)
print('Device      :', DEVICE)

Feature run : all_regions_v1_minside24_dinov2_vitb14_binarized_v1
Output      : <project>/2_clustering_outputs/all_regions_v1_minside24_dinov2_vitb14_binarized_v1_clustering_benchmark_v1
Device      : cuda


## 2. Load the selected feature matrix

We evaluate cosine similarity after L2 normalisation in the original 768-dimensional DINOv2 space. Scan identifiers come from the crop manifest. We infer book identifiers from the corpus filename convention and store them separately so they can be audited before we report any book-level claim.

The 58 identifiers printed below are the naive filename inference. They were later resolved against catalogue records into 57 physical volumes and 51 bibliographic works, in `book_identifiers.py`, and every book-level or work-level figure in this chapter uses the resolved counts rather than these.


In [2]:
feature_manifest = pd.read_csv(FEATURE_MANIFEST_PATH)
feature_matrix = np.load(FEATURES_PATH, mmap_mode='r')
assert feature_matrix.shape == (len(feature_manifest), 768)
assert feature_manifest.feature_index.tolist() == list(range(len(feature_manifest)))
assert feature_manifest.crop_path.map(lambda value: (PROJECT_DIR / value).exists()).all()

feature_norms = np.linalg.norm(feature_matrix, axis=1)
assert np.isfinite(feature_matrix).all() and (feature_norms > 0).all()
normalised_features = (
    np.asarray(feature_matrix) / feature_norms[:, None]
).astype(np.float32)

def infer_book_id(filename: str) -> str:
    stem = unicodedata.normalize('NFC', Path(filename).stem)
    if stem.upper().startswith('IMG_HDI'):
        return 'img_hdi'
    if stem.lower().startswith('vochroou70'):
        return 'vochroou70'
    return re.split(r'[.,_ \[]', stem)[0].lower()

feature_manifest['scan_id'] = feature_manifest.stem.astype(str)
feature_manifest['book_id'] = feature_manifest.filename.map(infer_book_id)

book_summary = (
    feature_manifest.groupby('book_id')
    .agg(
        scans=('scan_id', 'nunique'),
        candidates=('feature_index', 'size'),
        example_filename=('filename', 'first'),
    )
    .sort_values(['candidates', 'scans'], ascending=False)
    .reset_index()
)
book_summary.to_csv(BOOK_SUMMARY_PATH, index=False)

print('Feature vectors:', normalised_features.shape)
print('Scans          :', feature_manifest.scan_id.nunique())
print('Inferred books :', feature_manifest.book_id.nunique())
book_summary.head(10)

Feature vectors: (21750, 768)
Scans          : 612
Inferred books : 58


,book_id,scans,candidates,example_filename
0,dicolosn73,39,4668,dicolosn73.A3.t1.1.jpg
1,moœurovi65,50,2177,moœurovi65.nu.t1. lxxvi.bmp
2,vaoe1lhgo1760,54,1561,vaoe1lhgo1760_005.jpg
3,vochroou70,14,1128,vochroou70.ph.101.jpg
4,crcolonsn72,12,932,"crcolonsn72.t2,11.bmp"
5,vaoe4lhgo1761,40,892,vaoe4lhgo1761_041.jpg
6,gemorovi62,17,886,gemorovi62.nu (1).bmp
7,vaoe2lhgo1761,32,869,vaoe2lhgo1761_017.jpg
8,vohigecr60,18,802,vohigecr60.138.jpg
9,vaoe3lhgo1761,22,781,vaoe3lhgo1761_174.jpg


## 3. Sample blinded review pairs

Pairs are sampled from six cosine-similarity ranges. Random cross-scan pairs give an
algorithm-independent pool, while nearest-neighbour queries supply enough very-high-similarity
examples. The final review set holds 30 pairs per range and is shuffled, so that the reviewer sees
neither similarity scores nor calibration/evaluation membership.

In [3]:
scan_ids = feature_manifest.scan_id.to_numpy()
book_ids = feature_manifest.book_id.to_numpy()
candidate_frames = []

random_left = random_generator.integers(0, len(feature_manifest), size=RANDOM_PAIR_POOL)
random_right = random_generator.integers(0, len(feature_manifest), size=RANDOM_PAIR_POOL)
left_ordered = np.minimum(random_left, random_right)
right_ordered = np.maximum(random_left, random_right)
valid_random = (
    (left_ordered != right_ordered)
    & (scan_ids[left_ordered] != scan_ids[right_ordered])
)
left_ordered = left_ordered[valid_random]
right_ordered = right_ordered[valid_random]

kept_left, kept_right, kept_similarity = [], [], []
score_batch_size = 10_000
for start in tqdm(range(0, len(left_ordered), score_batch_size), desc='Random pair scores'):
    stop = min(start + score_batch_size, len(left_ordered))
    left_batch = left_ordered[start:stop]
    right_batch = right_ordered[start:stop]
    similarity_batch = np.einsum(
        'ij,ij->i', normalised_features[left_batch], normalised_features[right_batch]
    )
    keep_batch = similarity_batch >= SIMILARITY_BINS[0][1]
    kept_left.append(left_batch[keep_batch])
    kept_right.append(right_batch[keep_batch])
    kept_similarity.append(similarity_batch[keep_batch])

candidate_frames.append(pd.DataFrame({
    'left_index': np.concatenate(kept_left),
    'right_index': np.concatenate(kept_right),
    'similarity': np.concatenate(kept_similarity),
    'sampling_origin': 'random_pair',
}))

query_indices = random_generator.choice(
    len(feature_manifest), size=min(QUERY_COUNT, len(feature_manifest)), replace=False
)
if DEVICE == 'cuda':
    feature_tensor = torch.from_numpy(normalised_features).to(DEVICE)
    neighbor_rows = []
    query_batch_size = 256
    for start in tqdm(range(0, len(query_indices), query_batch_size), desc='Nearest neighbours'):
        query_batch = query_indices[start:start + query_batch_size]
        query_tensor = feature_tensor[torch.as_tensor(query_batch, device=DEVICE)]
        similarities = query_tensor @ feature_tensor.T
        similarities[
            torch.arange(len(query_batch), device=DEVICE),
            torch.as_tensor(query_batch, device=DEVICE),
        ] = -torch.inf
        values, indices = torch.topk(
            similarities, k=NEIGHBORS_PER_QUERY, dim=1
        )
        neighbor_rows.append((
            np.repeat(query_batch, NEIGHBORS_PER_QUERY),
            indices.cpu().numpy().reshape(-1),
            values.cpu().numpy().reshape(-1),
        ))
    del feature_tensor
    nearest_left = np.concatenate([row[0] for row in neighbor_rows])
    nearest_right = np.concatenate([row[1] for row in neighbor_rows])
    nearest_similarity = np.concatenate([row[2] for row in neighbor_rows])
else:
    from sklearn.neighbors import NearestNeighbors
    neighbor_model = NearestNeighbors(
        n_neighbors=NEIGHBORS_PER_QUERY + 1, metric='cosine', n_jobs=-1
    ).fit(normalised_features)
    distances, indices = neighbor_model.kneighbors(normalised_features[query_indices])
    nearest_left = np.repeat(query_indices, NEIGHBORS_PER_QUERY)
    nearest_right = indices[:, 1:].reshape(-1)
    nearest_similarity = (1.0 - distances[:, 1:]).reshape(-1)

nearest_first = np.minimum(nearest_left, nearest_right)
nearest_second = np.maximum(nearest_left, nearest_right)
valid_nearest = scan_ids[nearest_first] != scan_ids[nearest_second]
candidate_frames.append(pd.DataFrame({
    'left_index': nearest_first[valid_nearest],
    'right_index': nearest_second[valid_nearest],
    'similarity': nearest_similarity[valid_nearest],
    'sampling_origin': 'nearest_neighbor',
}))

pair_pool = (
    pd.concat(candidate_frames, ignore_index=True)
    .sort_values('similarity', ascending=False)
    .drop_duplicates(['left_index', 'right_index'])
    .reset_index(drop=True)
)

def similarity_bin(value: float):
    for label, lower, upper in SIMILARITY_BINS:
        if lower <= value < upper:
            return label
    return None

pair_pool['similarity_bin'] = pair_pool.similarity.map(similarity_bin)
pair_pool = pair_pool[pair_pool.similarity_bin.notna()].copy()
pair_pool['book_relation'] = np.where(
    book_ids[pair_pool.left_index.to_numpy(dtype=int)]
    == book_ids[pair_pool.right_index.to_numpy(dtype=int)],
    'same_book', 'cross_book',
)

pool_summary = (
    pair_pool.groupby(['similarity_bin', 'book_relation'], observed=False)
    .size().rename('available_pairs').reset_index()
)
pool_summary

,similarity_bin,book_relation,available_pairs
0,0.80-0.85,cross_book,34898
1,0.80-0.85,same_book,5999
2,0.85-0.90,cross_book,52026
3,0.85-0.90,same_book,13791
4,0.90-0.93,cross_book,26889
5,0.90-0.93,same_book,10879
6,0.93-0.95,cross_book,11342
7,0.93-0.95,same_book,6127
8,0.95-0.97,cross_book,8072
9,0.95-0.97,same_book,5185


In [4]:
selected_parts = []
for bin_label, _, _ in SIMILARITY_BINS:
    bin_pool = pair_pool[pair_pool.similarity_bin == bin_label]
    assert len(bin_pool) >= PAIRS_PER_BIN, f'Insufficient pairs in {bin_label}'
    relation_parts = []
    relation_target = PAIRS_PER_BIN // 2
    for relation in ['cross_book', 'same_book']:
        relation_pool = bin_pool[bin_pool.book_relation == relation]
        take = min(relation_target, len(relation_pool))
        if take:
            relation_parts.append(relation_pool.sample(
                take, random_state=RANDOM_SEED + len(selected_parts) + len(relation_parts)
            ))
    selected_bin = pd.concat(relation_parts) if relation_parts else bin_pool.iloc[0:0]
    remaining = bin_pool.drop(index=selected_bin.index)
    if len(selected_bin) < PAIRS_PER_BIN:
        selected_bin = pd.concat([
            selected_bin,
            remaining.sample(
                PAIRS_PER_BIN - len(selected_bin),
                random_state=RANDOM_SEED + 100 + len(selected_parts),
            ),
        ])
    selected_bin = selected_bin.sample(frac=1, random_state=RANDOM_SEED)
    selected_bin['benchmark_split'] = (
        ['calibration'] * CALIBRATION_PAIRS_PER_BIN
        + ['evaluation'] * (PAIRS_PER_BIN - CALIBRATION_PAIRS_PER_BIN)
    )
    selected_parts.append(selected_bin)

review_pairs = (
    pd.concat(selected_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED + 500)
    .reset_index(drop=True)
)
review_pairs.insert(0, 'pair_id', [f'P{index + 1:03d}' for index in range(len(review_pairs))])
review_pairs['sheet'] = np.arange(len(review_pairs)) // PAIRS_PER_SHEET + 1

manifest_lookup = feature_manifest.set_index('feature_index')
for side in ['left', 'right']:
    indices = review_pairs[f'{side}_index'].to_numpy(dtype=int)
    rows = manifest_lookup.loc[indices]
    for column in ['crop_path', 'filename', 'scan_id', 'book_id', 'width', 'height']:
        review_pairs[f'{side}_{column}'] = rows[column].to_numpy()

review_pairs.to_csv(PAIR_CANDIDATES_PATH, index=False)

review_form = review_pairs[['pair_id', 'sheet']].copy()
review_form['label'] = ''
review_form['confidence'] = ''
review_form['notes'] = ''
if PAIR_LABELS_PATH.exists():
    existing_form = pd.read_csv(PAIR_LABELS_PATH, keep_default_na=False)
    assert existing_form.pair_id.tolist() == review_form.pair_id.tolist()
    print('Existing review labels preserved.')
else:
    review_form.to_csv(PAIR_LABELS_PATH, index=False)

sampling_summary = (
    review_pairs.groupby(
        ['similarity_bin', 'book_relation', 'benchmark_split'], observed=False
    ).size().rename('pairs').reset_index()
)
sampling_summary.to_csv(PAIR_SUMMARY_PATH, index=False)

print(f'Review pairs: {len(review_pairs)}')
print(f'Calibration : {(review_pairs.benchmark_split == "calibration").sum()}')
print(f'Evaluation  : {(review_pairs.benchmark_split == "evaluation").sum()}')
sampling_summary

Existing review labels preserved.
Review pairs: 180
Calibration : 120
Evaluation  : 60


,similarity_bin,book_relation,benchmark_split,pairs
0,0.80-0.85,cross_book,calibration,11
1,0.80-0.85,cross_book,evaluation,4
2,0.80-0.85,same_book,calibration,9
3,0.80-0.85,same_book,evaluation,6
4,0.85-0.90,cross_book,calibration,11
5,0.85-0.90,cross_book,evaluation,4
6,0.85-0.90,same_book,calibration,9
7,0.85-0.90,same_book,evaluation,6
8,0.90-0.93,cross_book,calibration,11
9,0.90-0.93,cross_book,evaluation,4


## 4. Blinded review sheets

The 180 sampled pairs were rendered to contact sheets and reviewed by hand. Sheets showed only the
two crops and a pair identifier, no cosine similarity value, and no indication of calibration or
evaluation membership, so that the reviewer could not infer the expected answer or which pairs
would be used for tuning.

The rendering code is retained in the project history but is not re-executed here: regenerating the
sheets would produce a fresh, unlabelled review task and discard 180 completed judgements. The
sections below read the completed labels from `pair_review/pair_review_labels.csv`.

In [5]:
# Completed review labels: read-only summary of the human reference.
completed = pd.read_csv(OUTPUT_DIR / 'pair_review' / 'pair_review_labels.csv',
                        keep_default_na=False)
sampled = pd.read_csv(OUTPUT_DIR / 'pair_review' / 'pair_candidates.csv')
review = sampled.merge(completed[['pair_id', 'label']], on='pair_id')

print(f'pairs sampled : {len(sampled)}')
print(f'pairs labelled: {(review.label != "").sum()}')
print()
print(review.label.value_counts().to_string())
print()
print(pd.crosstab(review.benchmark_split, review.label).to_string())

pairs sampled : 180
pairs labelled: 180

label
same_fleuron    134
non_fleuron      29
different        16
unclear           1

label            different  non_fleuron  same_fleuron  unclear
benchmark_split                                               
calibration             11           16            92        1
evaluation               5           13            42        0


## 5. Status of the completed benchmark

The completed labels in `pair_review_labels.csv` form the human reference used in `3_MethodSelection.ipynb`, which compares HDBSCAN against mutual-kNN core-and-centroid grouping. Parameters were selected on the 120 calibration pairs alone; the 60 evaluation pairs were first opened after freezing selection.

The notebook records the split and applies the encoded selection rule before evaluation. The available historical protocol does not contain the exact numerical gates and tie-breakers, so the record supports the claim that the rule was encoded and applied as shown rather than the stronger claim that it was fixed prospectively in that exact form. Coverage, cluster-size behaviour, and non-fleuron grouping are reported beside the pair metrics rather than folded into the identity labels.

One design limitation of this notebook carries into every result that depends on it. We sampled pairs by cosine similarity band, which yields an overwhelming majority of positive pairs and only 16 negatives, eleven of them in the calibration split, and we split into calibration and evaluation by pair rather than by crop, so five crops occur in both splits. Both consequences are quantified in this chapter's README, §3 and §7 respectively, and `3_MethodSelection.ipynb` §7 measures what the eleven calibration negatives were able to decide. The disagreement benchmark of `4_MethodComparison.ipynb` supplies 35 negatives drawn where the candidate methods actually differ, but it was built after this selection was frozen and cannot retroactively repair it. A future selection benchmark should split on crops and deliberately include confusable negatives sampled independently of the representation being compared.

**What this notebook establishes.** It creates the small, blinded reference required to freeze a method choice: 180 reviewed pairs, of which 120 are calibration and 60 evaluation. It does not establish corpus-wide precision, cluster purity, or catalogue validity. Those are different estimands, measured later by the disagreement benchmark of `4_MethodComparison.ipynb` and the within-cluster audit of `6_WithinClusterAudit.ipynb`. `7_RepresentationCheck.ipynb` also reads these labels, so its comparison of descriptors rests on the same 16 negatives and inherits the same limit.